# Digit KNN Classifier — Decell Chicken Constructs 2026-04-15

**Why KNN instead of EasyOCR?** EasyOCR misreads ~10% of frames. A K-NN trained
on a handful of labeled display-crop images achieves much higher accuracy.

**Approach:**
1. Each experiment has its own Force Out display crop (the camera was in a
   different position per run) and its own `bright_x` region and `n_digits`
   count — digit column boxes are computed as contiguous equal-width slices.
2. Sample ~12 frames per video; pre-label with EasyOCR; save to
   `digit_crops/to_label/` for review.
3. **[User step]** Correct wrong filenames; delete unreadable XXXXX files.
4. Fit PCA + K-NN on the labeled crops.
5. Classify every frame of all 4 videos; export `_force_knn.csv` per video.
6. Merge with current CSVs and plot.

**Videos:**

| Tag | File | Force Out range | n_digits |
|-----|------|------------------|----------|
| `baseline`     | `2026.4.15-pyrrole-baseline.mov`        | ~7.6 V   | 4 |
| `decell_1soak` | `2026.4.15-decell_chicken_1_soak.mov`   | ~11.7 V  | 5 |
| `decell_3soak` | `2026.4.15-decell_chicken_3_soak.mov`   | ~3.2 V   | 4 |
| `decell_fresh` | `2026.4.15-decell_chicken_fresh.mov`    | ~3.2 V   | 4 |


In [ ]:
# ── CONFIG — each experiment has its own crop + HARDCODED digit_cols ─────────
from pathlib import Path
import numpy as np

DATA_DIR  = Path('actuation_currents_data')
LABEL_DIR = Path('digit_crops/to_label')
LABEL_DIR.mkdir(parents=True, exist_ok=True)

# Per-experiment configuration.
#   crop       = Force Out display region in the 1620×1080 frame
#   digit_cols = list of (x0, x1) per digit position, HAND-TUNED to cover each
#                digit generously (boxes may overlap slightly)
#   n_digits   = must equal len(digit_cols); also defines filename label length
#                (4 for readings <10 V, 5 for readings ≥10 V)
EXPERIMENTS = {
    'baseline': dict(
        video         = DATA_DIR / '2026.4.15-pyrrole-baseline.mov',
        current_csv   = DATA_DIR / '2026.04.15-pyrroleDBS-baselineactuation.csv',
        force_knn_csv = DATA_DIR / '2026.4.15-pyrrole-baseline_force_knn.csv',
        merged_csv    = DATA_DIR / '2026.4.15-pyrrole-baseline_merged_knn.csv',
        crop          = dict(y0=790, y1=884, x0=1175, x1=1415),
        digit_cols    = [(14, 66), (57, 110), (101, 153), (144, 197)],
        n_digits      = 4,
        v_min=6.5, v_max=9.0,
        interval_s=120,  video_offset_s=0.0,  color='steelblue',
    ),
    'decell_1soak': dict(
        video         = DATA_DIR / '2026.4.15-decell_chicken_1_soak.mov',
        current_csv   = DATA_DIR / '2026.4.15-decel-chicken-1soakactuation.csv',
        force_knn_csv = DATA_DIR / '2026.4.15-decell_chicken_1_soak_force_knn.csv',
        merged_csv    = DATA_DIR / '2026.4.15-decell_chicken_1_soak_merged_knn.csv',
        crop          = dict(y0=812, y1=909, x0=1095, x1=1372),
        digit_cols    = [(7, 61), (52, 107), (98, 153), (143, 198), (189, 244)],
        n_digits      = 5,
        v_min=9.0, v_max=14.0,
        interval_s=200,  video_offset_s=0.0,  color='seagreen',
    ),
    'decell_3soak': dict(
        video         = DATA_DIR / '2026.4.15-decell_chicken_3_soak.mov',
        current_csv   = DATA_DIR / '2026.4.15-decel-chicken-3soakactuation.csv',
        force_knn_csv = DATA_DIR / '2026.4.15-decell_chicken_3_soak_force_knn.csv',
        merged_csv    = DATA_DIR / '2026.4.15-decell_chicken_3_soak_merged_knn.csv',
        crop          = dict(y0=692, y1=782, x0=1120, x1=1392),
        digit_cols    = [(35, 91), (82, 137), (128, 184), (175, 230)],
        n_digits      = 4,
        v_min=1.0, v_max=6.0,
        interval_s=200,  video_offset_s=0.0,  color='darkorange',
    ),
    'decell_fresh': dict(
        video         = DATA_DIR / '2026.4.15-decell_chicken_fresh.mov',
        current_csv   = DATA_DIR / '2026.4.15-decel-chicken-fresh-actuation.csv',
        force_knn_csv = DATA_DIR / '2026.4.15-decell_chicken_fresh_force_knn.csv',
        merged_csv    = DATA_DIR / '2026.4.15-decell_chicken_fresh_merged_knn.csv',
        crop          = dict(y0=819, y1=907, x0=1010, x1=1285),
        digit_cols    = [(62, 112), (104, 153), (145, 195), (187, 236)],
        n_digits      = 4,
        v_min=1.0, v_max=6.0,
        interval_s=200,  video_offset_s=0.0,  color='mediumpurple',
    ),
}


def get_digit_cols(cfg):
    """Return hardcoded digit_cols (sanity-check length matches n_digits)."""
    cols = cfg['digit_cols']
    assert len(cols) == cfg['n_digits'], \
        f"digit_cols has {len(cols)} entries but n_digits={cfg['n_digits']}"
    return cols


# ── KNN / PCA ─────────────────────────────────────────────────────────────────
CROP_W, CROP_H    = 44, 88
N_COMPONENTS      = 20
N_NEIGHBORS       = 3
UNSURE_MULTIPLIER = 2.5

DIGIT_CONSTRAINTS = {0: None, 1: None, 2: None, 3: None, 4: None}

# ── Sampling ──────────────────────────────────────────────────────────────────
SAMPLE_N_PER_VIDEO = 12
OCR_DT_S           = 0.5

# ── Voltage assembly: insert decimal 3 from right ────────────────────────────
DECIMAL_FROM_RIGHT = 3


def digits_to_voltage(digits):
    """[d0, d1, ...] (int) → voltage float; decimal inserted 3 from the right."""
    if any(d is None for d in digits):
        return None
    s = ''.join(str(d) for d in digits)
    int_part = s[:-DECIMAL_FROM_RIGHT] or '0'
    return float(f'{int_part}.{s[-DECIMAL_FROM_RIGHT:]}')


# ── Force calibration (matches analysis.ipynb) ────────────────────────────────
FORCE_SCALE_mN_per_V = 56
FORCE_OFFSET_V       = 0.0
APPLIED_V_AMPLITUDE  = 0.8

print('Config loaded.')
for tag, cfg in EXPERIMENTS.items():
    c = cfg['crop']
    cols = get_digit_cols(cfg)
    widths = [b-a for a,b in cols]
    print(f'  {tag:14s}  crop {c["x1"]-c["x0"]}×{c["y1"]-c["y0"]}  '
          f'n_digits={cfg["n_digits"]}  widths={widths}')
    print(f'                  digit_cols={cols}')


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import easyocr
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from scipy.signal import medfilt
from tqdm.auto import tqdm
from pathlib import Path

ocr_reader = easyocr.Reader(['en'], gpu=False, verbose=False)
print('Imports OK')


## Step 1 — Verify per-experiment crops and digit columns

Each experiment has its own Force Out display crop (camera position varies)
and its own `bright_x` region. The cell below shows, for each experiment,
three reference frames with the **contiguous** digit boxes overlaid.

**If any box misses the real digit region**, edit `bright_x=(x_start, x_end)`
or `n_digits` in the config cell above and re-run.


In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────
def get_display_crop_bgr(frame, cfg):
    c = cfg['crop']
    return frame[c['y0']:c['y1'], c['x0']:c['x1']]


def get_display_crop_gray(frame, cfg):
    return cv2.cvtColor(get_display_crop_bgr(frame, cfg), cv2.COLOR_BGR2GRAY)


def threshold_display(gray):
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresh


def extract_digit_cell(display_gray_or_thresh, x0, x1, out_w=CROP_W, out_h=CROP_H):
    h, w = display_gray_or_thresh.shape[:2]
    x0 = max(0, x0); x1 = min(w, x1)
    cell = display_gray_or_thresh[:, x0:x1]
    return cv2.resize(cell, (out_w, out_h), interpolation=cv2.INTER_NEAREST)


# ── Show 3 reference frames per experiment with contiguous boxes ──────────────
BOX_COLORS_RGB = [(0,200,0), (255,160,0), (0,130,255), (200,0,200), (200,200,0)]

n_exp = len(EXPERIMENTS)
fig, axes = plt.subplots(n_exp, 3, figsize=(15, 2.4 * n_exp))
if n_exp == 1: axes = axes[np.newaxis, :]

for row, (tag, cfg) in enumerate(EXPERIMENTS.items()):
    cap = cv2.VideoCapture(str(cfg['video']))
    dur = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
    digit_cols = get_digit_cols(cfg)
    for col, frac in enumerate([0.2, 0.5, 0.8]):
        cap.set(cv2.CAP_PROP_POS_MSEC, dur * frac * 1000)
        ret, frame = cap.read()
        if not ret: continue
        crop = get_display_crop_bgr(frame, cfg).copy()
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        ax = axes[row, col]
        ax.imshow(crop_rgb)
        for i, (x0, x1) in enumerate(digit_cols):
            col_rgb = BOX_COLORS_RGB[i]
            rect = plt.Rectangle((x0, 1), x1-x0-1, crop.shape[0]-3,
                                 fill=False, edgecolor=np.array(col_rgb)/255, lw=2)
            ax.add_patch(rect)
        ax.set_title(f'{tag}  |  t={frac*dur:.0f}s', fontsize=9)
        ax.axis('off')
    cap.release()

plt.suptitle('Per-experiment Force Out crops with contiguous digit boxes  '
             '(adjust bright_x / n_digits in config if needed)', y=1.005)
plt.tight_layout()
plt.show()


## Step 2 — Sample frames and pre-label with EasyOCR

For each video we sample `SAMPLE_N_PER_VIDEO` frames, run EasyOCR on the display
crop to produce a guessed label, and save the crop to `digit_crops/to_label/`
with the label baked into the filename.

**Filename format:** `{digits}_{tag}_{idx:03d}.png`  
→ baseline 4-digit labels: `7615_baseline_000.png`  
→ 1-soak 5-digit labels: `11742_decell_1soak_000.png`

After this cell runs, open `digit_crops/to_label/` and rename any file where
the leading digits don't match the displayed reading.


In [ ]:
def ocr_display_crop(crop_bgr, n_digits, v_min, v_max):
    """Run EasyOCR; return digit string of exactly n_digits chars (or None)."""
    result = ocr_reader.readtext(crop_bgr, detail=1, allowlist='0123456789.')
    if not result: return None
    best = max(result, key=lambda r: r[2])
    digits_only = best[1].replace('.', '').strip()
    if not digits_only.isdigit(): return None

    def assemble(d):
        i = d[:-DECIMAL_FROM_RIGHT] or '0'
        return float(f'{i}.{d[-DECIMAL_FROM_RIGHT:]}')

    # Exact-length match
    if len(digits_only) == n_digits:
        if v_min <= assemble(digits_only) <= v_max:
            return digits_only

    # Try prepending a digit if OCR dropped the leading one
    if len(digits_only) == n_digits - 1:
        for pref in ['0', '1']:
            candidate = pref + digits_only
            if v_min <= assemble(candidate) <= v_max:
                return candidate

    # Try truncating leading char if OCR added one
    if len(digits_only) == n_digits + 1:
        if v_min <= assemble(digits_only[1:]) <= v_max:
            return digits_only[1:]

    return None


def sample_and_label(tag, cfg, n_samples=SAMPLE_N_PER_VIDEO):
    cap = cv2.VideoCapture(str(cfg['video']))
    dur = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
    times = np.linspace(0.05 * dur, 0.95 * dur, n_samples)
    saved, failed = 0, 0
    for idx, t in enumerate(times):
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ret, frame = cap.read()
        if not ret: failed += 1; continue
        crop_bgr = get_display_crop_bgr(frame, cfg)
        label = ocr_display_crop(crop_bgr, cfg['n_digits'], cfg['v_min'], cfg['v_max'])
        if label is None:
            label = 'X' * cfg['n_digits']
            failed += 1
        fname = LABEL_DIR / f'{label}_{tag}_{idx:03d}.png'
        cv2.imwrite(str(fname), crop_bgr)
        saved += 1
    cap.release()
    print(f'  [{tag}] saved {saved} crops ({failed} OCR failures → label = {"X"*cfg["n_digits"]})')


print('Sampling and OCR-labeling frames...')
for tag, cfg in EXPERIMENTS.items():
    sample_and_label(tag, cfg)

all_crops = sorted(LABEL_DIR.glob('*.png'))
print(f'\nTotal files in digit_crops/to_label/: {len(all_crops)}')
print('→ Review the folder, rename any file with a wrong leading-digit label.')


In [ ]:
# ── Visualize all labeled crops with digit-box overlays ───────────────────────
# Each image shows the display crop AS IT WAS SAVED (no boxes in the saved file)
# plus the digit boxes overlaid as colored rectangles, so you can see exactly
# which pixels the KNN will use for each digit position.

def filename_to_label_and_tag(path):
    """'{digits}_{tag}_{idx}.png' → (digit_list, tag) or (None, None)."""
    stem = path.stem
    first_us = stem.find('_')
    if first_us < 0: return None, None
    label = stem[:first_us]
    if not label.isdigit(): return None, None
    rest = stem[first_us+1:]
    last_us = rest.rfind('_')
    tag = rest[:last_us] if last_us > 0 else rest
    if tag not in EXPERIMENTS: return None, None
    cfg = EXPERIMENTS[tag]
    if len(label) != cfg['n_digits']: return None, None
    return [int(c) for c in label], tag


def tag_of(path):
    """Return tag for a file whose label may be invalid (e.g. XXXX)."""
    stem = path.stem
    first_us = stem.find('_')
    if first_us < 0: return None
    rest = stem[first_us+1:]
    last_us = rest.rfind('_')
    tag = rest[:last_us] if last_us > 0 else rest
    return tag if tag in EXPERIMENTS else None


# Show EVERY file (valid label or XXXX), grouped by experiment
BOX_COLORS_RGB = [(0, 220, 0), (255, 160, 0), (50, 130, 255), (220, 0, 220), (230, 230, 0)]

all_files = sorted(LABEL_DIR.glob('*.png'))
print(f'{len(all_files)} files in digit_crops/to_label/')

# Group by experiment so each column lines up
by_tag = {tag: [] for tag in EXPERIMENTS}
for p in all_files:
    tag = tag_of(p)
    if tag is not None:
        by_tag[tag].append(p)

max_per_tag = max(len(v) for v in by_tag.values()) if any(by_tag.values()) else 0
if max_per_tag == 0:
    print('No valid files to display.')
else:
    ncols = len(EXPERIMENTS)
    nrows = max_per_tag
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 1.4))
    if nrows == 1: axes = axes[np.newaxis, :]

    for col, (tag, paths) in enumerate(by_tag.items()):
        cfg = EXPERIMENTS[tag]
        digit_cols = get_digit_cols(cfg)
        for row in range(nrows):
            ax = axes[row, col]
            if row >= len(paths):
                ax.axis('off'); continue
            p = paths[row]
            img = cv2.imread(str(p))
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            # Overlay boxes
            for i, (x0, x1) in enumerate(digit_cols):
                c_rgb = np.array(BOX_COLORS_RGB[i]) / 255
                rect = plt.Rectangle((x0, 1), x1-x0-1, img.shape[0]-3,
                                     fill=False, edgecolor=c_rgb, lw=1.5)
                ax.add_patch(rect)
            # Title with filename label + assembled voltage (if valid)
            digs, _ = filename_to_label_and_tag(p)
            if digs is not None:
                v = digits_to_voltage(digs)
                ttl = f'{"".join(map(str,digs))}  ({v:.3f} V)'
            else:
                ttl = p.stem.split("_")[0]   # e.g. "XXXX"
            ax.set_title(f'{ttl}\n{p.name}', fontsize=6)
            ax.axis('off')

    for col, tag in enumerate(by_tag):
        axes[0, col].set_title(f'[{tag}]  n={len(by_tag[tag])}\n' + axes[0, col].get_title(),
                               fontsize=7, loc='center')
    plt.suptitle('digit_crops/to_label/  —  boxes show exactly which pixels each digit uses', y=1.002)
    plt.tight_layout()
    plt.show()


## Step 3 — [User review]

Before continuing:

1. Open `digit_crops/to_label/` in Finder.
2. For each image, compare the 7-segment reading to the **leading digits** of the filename.
3. If wrong, rename the file so the leading digits match (keep the `_tag_idx.png` suffix).
4. Delete any file whose label is `XXXX` / `XXXXX` and you cannot read it.
5. Re-run the visualization cell above to confirm, then continue.


## Step 4 — Fit PCA + K-NN

Extract individual digit crops from each labeled display crop using its
experiment's digit_cols, then train the classifier.


In [ ]:
X_train, y_train, P_train = [], [], []
valid = []
for p in sorted(LABEL_DIR.glob('*.png')):
    digs, tag = filename_to_label_and_tag(p)
    if digs is not None: valid.append((p, digs, tag))

print(f'Loading {len(valid)} labeled crops...')
for p, digits, tag in valid:
    cfg = EXPERIMENTS[tag]
    digit_cols = get_digit_cols(cfg)
    img_bgr = cv2.imread(str(p))
    if img_bgr is None: continue
    gray   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    thresh = threshold_display(gray)
    for pos, (x0, x1) in enumerate(digit_cols):
        cell = extract_digit_cell(thresh, x0, x1)
        X_train.append(cell.flatten().astype(np.float32) / 255.0)
        y_train.append(digits[pos])
        P_train.append(pos)

X_train = np.array(X_train); y_train = np.array(y_train); P_train = np.array(P_train)
print(f'Training crops: {len(X_train)}')
for d in sorted(set(y_train)):
    print(f'  digit {d}: {(y_train==d).sum()} examples')

n_comp = min(N_COMPONENTS, len(X_train) - 1)
pca = PCA(n_components=n_comp, whiten=True, random_state=0)
X_pca = pca.fit_transform(X_train)
knn = KNeighborsClassifier(n_neighbors=min(N_NEIGHBORS, len(X_train)), metric='euclidean')
knn.fit(X_pca, y_train)
print(f'\nPCA: {pca.n_components_} components, '
      f'{pca.explained_variance_ratio_.sum()*100:.1f}% variance explained')

cal_dists, _ = knn.kneighbors(X_pca, n_neighbors=min(N_NEIGHBORS+1, len(X_train)))
nn_dists = cal_dists[:, 1]
UNSURE_THRESHOLD = np.percentile(nn_dists, 90) * UNSURE_MULTIPLIER
print(f'Unsure threshold: {UNSURE_THRESHOLD:.4f} (×{UNSURE_MULTIPLIER})')

fig, ax = plt.subplots(figsize=(9, 6))
for d in sorted(set(y_train)):
    m = y_train == d
    ax.scatter(X_pca[m, 0], X_pca[m, 1], label=str(d), s=60, alpha=0.8)
ax.set(xlabel='PC1', ylabel='PC2', title='PCA of labeled digit crops')
ax.legend(title='digit', ncol=2)
plt.tight_layout(); plt.show()


## Step 5 — Classify all video frames

For each video, sample every `OCR_DT_S` seconds, classify each digit with the
K-NN using the experiment's digit_cols, and fall back to the last known-good
reading when any digit is unsure.


In [ ]:
def classify_frame_digits(display_gray, digit_cols):
    thresh = threshold_display(display_gray)
    crops = [extract_digit_cell(thresh, x0, x1).flatten().astype(np.float32)/255.0
             for x0, x1 in digit_cols]
    X_p = pca.transform(np.array(crops))
    results = []
    for pos in range(len(digit_cols)):
        allowed = DIGIT_CONSTRAINTS.get(pos)
        dists, idxs = knn.kneighbors(X_p[pos:pos+1], n_neighbors=len(X_train))
        dists, idxs = dists[0], idxs[0]
        if allowed is not None:
            mask = np.array([y_train[i] in set(allowed) for i in idxs])
            dists, idxs = dists[mask], idxs[mask]
        if len(dists) == 0:
            results.append(None); continue
        results.append(None if dists[0] > UNSURE_THRESHOLD else int(y_train[idxs[0]]))
    return results


def digits_to_force_v(digits, v_min, v_max):
    if any(d is None for d in digits): return None
    v = digits_to_voltage(digits)
    return v if (v is not None and v_min <= v <= v_max) else None


def classify_video(tag, cfg):
    out_path = cfg['force_knn_csv']
    if out_path.exists():
        print(f'[{tag}] {out_path.name} exists — loading cache. Delete to re-run.')
        return pd.read_csv(out_path)
    digit_cols = get_digit_cols(cfg)
    cap = cv2.VideoCapture(str(cfg['video']))
    dur = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
    timestamps = np.arange(0, dur, OCR_DT_S)
    print(f'[{tag}] {dur:.0f} s  |  {len(timestamps)} frames')
    records, last_good = [], None
    for t in tqdm(timestamps, desc=tag):
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ret, frame = cap.read()
        if not ret:
            records.append({'time_s': round(t,3), 'force_V': last_good,
                            'confident': False, 'digits': str([None]*cfg['n_digits'])})
            continue
        gray   = get_display_crop_gray(frame, cfg)
        digits = classify_frame_digits(gray, digit_cols)
        v      = digits_to_force_v(digits, cfg['v_min'], cfg['v_max'])
        if v is not None: last_good = v
        records.append({'time_s': round(t,3),
                        'force_V': v if v is not None else last_good,
                        'confident': v is not None, 'digits': str(digits)})
    cap.release()
    df = pd.DataFrame(records)
    n_ok = df['confident'].sum()
    print(f'  Confident: {n_ok} ({100*n_ok/len(df):.1f}%)')
    df.to_csv(out_path, index=False)
    print(f'  Saved → {out_path.name}')
    return df


force_dfs = {tag: classify_video(tag, cfg) for tag, cfg in EXPERIMENTS.items()}


## Step 6 — Diagnostic plots

Spot-check each video's raw KNN output before merging.


In [ ]:
fig, axes = plt.subplots(len(EXPERIMENTS), 1, figsize=(14, 3 * len(EXPERIMENTS)))
if len(EXPERIMENTS) == 1: axes = [axes]
for ax, (tag, cfg) in zip(axes, EXPERIMENTS.items()):
    df = force_dfs[tag]
    conf = df[df['confident']]
    ax.scatter(conf['time_s'], conf['force_V'], s=1.5, color=cfg['color'], alpha=0.6)
    ax.set(ylabel='Force Out (V)', title=f'{tag} — raw KNN (confident only)')
    ax.grid(True, lw=0.3, alpha=0.4)
axes[-1].set_xlabel('Video time (s)')
plt.tight_layout(); plt.show()


## Step 7 — Merge with current CSVs and final plots


In [ ]:
def load_current_csv(path):
    df = pd.read_csv(path, encoding='utf-16', skiprows=5)
    df.columns = ['time_s', 'current_uA']
    df['time_s']     = pd.to_numeric(df['time_s'], errors='coerce')
    df['current_uA'] = pd.to_numeric(df['current_uA'], errors='coerce')
    return df.dropna().reset_index(drop=True)


def make_applied_voltage(time_s, interval_s, amplitude=APPLIED_V_AMPLITUDE):
    cycle = (time_s // interval_s).astype(int)
    return np.where(cycle % 2 == 0, +amplitude, -amplitude)


merged_dfs = {}
for tag, cfg in EXPERIMENTS.items():
    df_cur = load_current_csv(cfg['current_csv'])
    df_cur['applied_V'] = make_applied_voltage(df_cur['time_s'].values, cfg['interval_s'])
    df_v = force_dfs[tag].copy()
    df_v['time_s'] = df_v['time_s'] + cfg['video_offset_s']
    merged = pd.merge_asof(
        df_cur.sort_values('time_s'),
        df_v.sort_values('time_s')[['time_s','force_V','confident']],
        on='time_s', direction='nearest', tolerance=1.0)
    merged['force_mN'] = (merged['force_V'] - FORCE_OFFSET_V) * FORCE_SCALE_mN_per_V
    merged.to_csv(cfg['merged_csv'], index=False)
    merged_dfs[tag] = merged
    print(f'[{tag}] merged {len(merged)} rows → {cfg["merged_csv"].name}')


In [ ]:
for tag, cfg in EXPERIMENTS.items():
    df = merged_dfs[tag]
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    axes[0].step(df['time_s'], df['applied_V'], lw=1.0, color='darkorange', where='post')
    axes[0].axhline(0, color='k', lw=0.5, ls='--')
    axes[0].set(ylabel='Applied V (V)',
                title=f'{tag}  |  ±{APPLIED_V_AMPLITUDE} V, {cfg["interval_s"]} s half-cycle')
    axes[1].plot(df['time_s'], df['current_uA'], lw=0.6, color='steelblue')
    axes[1].set(ylabel='Current (µA)')
    axes[2].plot(df['time_s'], df['force_mN'], lw=0.8, color=cfg['color'])
    axes[2].set(xlabel='Time (s)', ylabel='Force (mN)')
    for ax in axes: ax.grid(True, lw=0.3, alpha=0.4)
    plt.tight_layout(); plt.show()

decell_tags = [t for t in EXPERIMENTS if t != 'baseline']
fig, axes = plt.subplots(len(decell_tags), 1, figsize=(14, 4*len(decell_tags)))
if len(decell_tags) == 1: axes = [axes]
for ax, tag in zip(axes, decell_tags):
    df  = merged_dfs[tag]; cfg = EXPERIMENTS[tag]
    ax2 = ax.twinx()
    ax.plot(df['time_s'],  df['force_mN'],  lw=0.8, color=cfg['color'], label='Force (mN)')
    ax2.step(df['time_s'], df['applied_V'], lw=0.8, color='darkorange',
             alpha=0.5, where='post', label='Applied V')
    ax.set(ylabel='Force (mN)', title=tag)
    ax2.set(ylabel='Applied V (V)', ylim=(-2, 2))
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')
    ax.grid(True, lw=0.3, alpha=0.4)
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Decell chicken constructs — Force vs Applied Voltage', y=1.01)
plt.tight_layout(); plt.show()
